# CeNNMixer-v4 Direct-1024 — single-layer convergence experiment

This notebook removes the alpha/takeover curriculum completely. **Layer 0 is replaced by the compact CeNNMixer from the first training update.** The frozen native Qwen3.5 mixer is used only as a teacher target inside the wrapper.

### Mathematical changes

For the compact associative branch we now match Qwen3.5's Gated-Delta recurrence:

\[
g_t=-\exp(A)\,\mathrm{softplus}(a_t+dt),\qquad
S_t^- = \exp(g_t)S_{t-1},
\]

\[
e_t=v_t-k_t^\top S_t^-,\qquad
S_t=S_t^-+\beta_t k_t e_t^\top,\qquad
o_t=q_t^\top S_t.
\]

The query/key vectors are L2-normalized and \(q\) is scaled by \(1/\sqrt{d_k}\).

Instead of random initialization, each of Qwen3.5-0.8B's 16 native Gated-Delta heads is compressed into:
- a **joint dominant q/k subspace** (preserves dot-product geometry),
- a **joint dominant v/z subspace** (preserves memory values and output gating),
- a composed compressed output projection,
- copied beta and continuous-time decay projections.

The CeNN correction branch uses a contractive normalized neighbor operator and EMA-style multi-timescale state updates so its state remains stable over 1024-token windows.

### Training objective

There is no interpolation with Qwen. The active compact layer is optimized directly with:
1. relative mixer-output MSE,
2. cosine error,
3. multi-lag temporal error (1/4/16/64 tokens),
4. multiscale pooled error (8/32/128 tokens),
5. RMS/amplitude matching,
6. tail-window fidelity,
7. forward/reverse KL, top-k rank and top-1 margin losses,
8. hidden-state fidelity and a small language-model CE term.

Training uses **1024 tokens from update 1**. Validation checks 256, 512 and 1024 token contexts.

In [ ]:
#@title 1. Setup
import pathlib, subprocess, sys, importlib, json, torch

REPO_REF='main'
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM-v4-direct')
if not REPO_DIR.exists():
    subprocess.run([
        'git','clone','--branch',REPO_REF,
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)
    ],check=True)
else:
    dirty=subprocess.check_output(
        ['git','-C',str(REPO_DIR),'status','--porcelain'],text=True
    )
    if dirty.strip():
        raise RuntimeError('Checkout has local edits. Preserve them or choose another REPO_DIR.')
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',REPO_REF],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',REPO_REF],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/'+REPO_REF],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers==5.17.0','accelerate','datasets','pandas','matplotlib',
    'huggingface_hub','safetensors'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC=REPO_DIR/'src'
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tinycenn_lm' or name.startswith('tinycenn_lm.'):
        del sys.modules[name]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v4_core.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v4.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v4_train.py',
    REPO_DIR/'scripts'/'run_qwen35_cennmixer_v4.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('✓ Direct CeNNMixer-v4 preflight OK')
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun setup.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Configuration — one layer, 1024-token training
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
LAYER=0 #@param {type:'integer'}

# Full training context from the first update.
SEQ_LEN=1024 #@param {type:'integer'}
CONTEXT_LENGTHS='256,512,1024' #@param {type:'string'}
TRAIN_BLOCKS=768 #@param {type:'integer'}
VAL_BLOCKS=4 #@param {type:'integer'}

# Contractive CeNN correction.
GROUPS=24 #@param {type:'integer'}
CELL_DIM=32 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}

# Preserve Qwen's 16 Gated-Delta heads, compress 128 -> 64 dimensions/head.
ASSOC_HEADS=16 #@param {type:'integer'}
KEY_DIM=64 #@param {type:'integer'}
VALUE_DIM=64 #@param {type:'integer'}
CONV_KERNEL=4 #@param {type:'integer'}

LR=0.0001 #@param {type:'number'}
MIN_LR=0.00001 #@param {type:'number'}
WARMUP_UPDATES=50 #@param {type:'integer'}
UPDATES=1200 #@param {type:'integer'}
MIN_UPDATES=200 #@param {type:'integer'}
PROBE_EVERY=50 #@param {type:'integer'}
LOGIT_TOKENS=128 #@param {type:'integer'}
TOPK=32 #@param {type:'integer'}

ON_POLICY_EVERY=100 #@param {type:'integer'}
ON_POLICY_TOKENS=16 #@param {type:'integer'}

MIN_TOP1=0.97 #@param {type:'number'}
MAX_KL=0.03 #@param {type:'number'}
MAX_HIDDEN_MSE=0.05 #@param {type:'number'}
MAX_MIXER_MSE=0.12 #@param {type:'number'}
MAX_MIXER_COSINE=0.07 #@param {type:'number'}
MAX_MIXER_DELTA=0.20 #@param {type:'number'}
MAX_MIXER_MULTISCALE=0.12 #@param {type:'number'}
MAX_CE_GAP=0.05 #@param {type:'number'}

OUTPUT_DIR=pathlib.Path('/content/cennmixer_v4_direct_1024_results')

print('Layer:',LAYER)
print('Training context:',SEQ_LEN)
print('Evaluation contexts:',CONTEXT_LENGTHS)
print('Associative state:',ASSOC_HEADS,'×',KEY_DIM,'×',VALUE_DIM)
print('Runtime associative state floats:',ASSOC_HEADS*KEY_DIM*VALUE_DIM)
print('No alpha schedule: compact layer is active from update 1.')

In [ ]:
#@title 3. Start training — direct compact mixer, no alpha
cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_cennmixer_v4.py'),
    '--base-model',BASE_MODEL,
    '--layer',str(LAYER),
    '--seq-len',str(SEQ_LEN),
    '--context-lengths',CONTEXT_LENGTHS,
    '--train-blocks',str(TRAIN_BLOCKS),
    '--val-blocks',str(VAL_BLOCKS),
    '--groups',str(GROUPS),
    '--cell-dim',str(CELL_DIM),
    '--graph-steps',str(GRAPH_STEPS),
    '--assoc-heads',str(ASSOC_HEADS),
    '--key-dim',str(KEY_DIM),
    '--value-dim',str(VALUE_DIM),
    '--conv-kernel',str(CONV_KERNEL),
    '--lr',str(LR),
    '--min-lr',str(MIN_LR),
    '--warmup-updates',str(WARMUP_UPDATES),
    '--updates',str(UPDATES),
    '--min-updates',str(MIN_UPDATES),
    '--probe-every',str(PROBE_EVERY),
    '--logit-tokens',str(LOGIT_TOKENS),
    '--topk',str(TOPK),
    '--on-policy-every',str(ON_POLICY_EVERY),
    '--on-policy-tokens',str(ON_POLICY_TOKENS),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-hidden-mse',str(MAX_HIDDEN_MSE),
    '--max-mixer-mse',str(MAX_MIXER_MSE),
    '--max-mixer-cosine',str(MAX_MIXER_COSINE),
    '--max-mixer-delta',str(MAX_MIXER_DELTA),
    '--max-mixer-multiscale',str(MAX_MIXER_MULTISCALE),
    '--max-ce-gap',str(MAX_CE_GAP),
    '--output-dir',str(OUTPUT_DIR),
]

print('='*120)
print(' '.join(cmd))
print('='*120)

p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)

In [ ]:
#@title 4. Results
import pandas as pd
from IPython.display import display

report=json.loads((OUTPUT_DIR/'report.json').read_text())
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv') if (OUTPUT_DIR/'training_history.csv').exists() else pd.DataFrame()

print('Architecture:',report['architecture'])
print('Training mode:',report['training_mode'])
print('Layer:',report['layer'],report['layer_kind'])
print('Training context:',report['train_context'])
print('Evaluated contexts:',report['evaluated_context_lengths'])
print('Teacher-aligned init:',report['teacher_aligned_init'])
print('Best step:',report['best_step'])
print('Strict final quality:',report['strict_quality_gate'])
print('CeNN-v4 params:',f"{report['cenn_params']:,}")
print('Qwen mixer params:',f"{report['replaced_qwen_mixer_params']:,}")
print('Mixer parameter reduction:',f"{report['mixer_param_reduction_pct']:.2f}%")
print('Runtime associative state floats:',report['runtime_associative_state_floats'])

print('\nBest direct probe:')
display(pd.DataFrame([report['best_direct_probe']]))

print('\nFinal CeNN-only probe:')
display(pd.DataFrame([report['final_cenn_only_probe']]))

print('\nPer-context final metrics:')
display(pd.DataFrame(report['final_cenn_only_per_context']).T)

In [ ]:
#@title 5. Convergence curves
import matplotlib.pyplot as plt

if hist.empty:
    print('No training_history.csv yet.')
else:
    plt.figure(figsize=(9,5))
    plt.plot(hist['step'],hist['train_mixer_mse'],marker='o',label='train mixer MSE')
    if 'mixer_mse' in hist:
        plt.plot(hist['step'],hist['mixer_mse'],marker='o',label='validation mixer MSE')
    plt.yscale('log')
    plt.xlabel('update')
    plt.ylabel('relative error')
    plt.title('Direct mixer convergence at 1024-token training context')
    plt.legend()
    plt.show()

    plt.figure(figsize=(9,5))
    plt.plot(hist['step'],hist['kl'],marker='o',label='KL')
    plt.plot(hist['step'],hist['hidden_mse'],marker='o',label='hidden MSE')
    plt.xlabel('update')
    plt.ylabel('error')
    plt.title('End-to-end convergence to native Qwen')
    plt.legend()
    plt.show()

    plt.figure(figsize=(9,5))
    plt.plot(hist['step'],hist['top1'],marker='o')
    plt.axhline(MIN_TOP1,linestyle='--')
    plt.xlabel('update')
    plt.ylabel('top-1 agreement')
    plt.title('Token prediction agreement')
    plt.show()

In [ ]:
#@title 6. Inspect final generations
rows=json.loads((OUTPUT_DIR/'generation_final.json').read_text())
for i,x in enumerate(rows,1):
    print('\n'+'='*100)
    print(i,'USER:',x['prompt'])
    print('QWEN:',x['qwen'])
    print('CeNN-v4:',x['cenn'])
    print('exact=',x['exact'],'jaccard=',round(x['jaccard'],3),
          'student_correct=',x['student_correct'],
          'prompt_tokens=',x['prompt_tokens'])

### What counts as convergence?

The important comparison is now **direct compact vs. native Qwen**, not a blended model. A good run should show all of the following trending together:

- direct mixer relative MSE decreasing,
- multi-lag and multiscale errors decreasing,
- hidden-state MSE decreasing,
- KL decreasing,
- top-1 agreement increasing,
- no regression at the 1024-token validation window,
- healthy retrieval/generation after the frozen native mixer is physically removed.

The final report separates the best training-wrapper probe from the final CeNN-only probe so any hidden dependence on the teacher wrapper would be visible.